In [1]:
from langchain_aws import ChatBedrockConverse
from langchain_ollama import ChatOllama,OllamaEmbeddings
from typing import TypedDict, Literal,List,Dict,Annotated
from langgraph.graph import StateGraph,START,END,MessagesState
from pydantic import BaseModel,Field
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph.message import add_messages
import sqlite3
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage,BaseMessage,ToolMessage
import operator
from langchain_core.tools import tool
import uuid
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command,interrupt
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware,PIIMiddleware,SummarizationMiddleware
from langgraph.prebuilt import ToolNode, tools_condition
from langchain.tools import ToolException
from langchain_aws import BedrockEmbeddings
from langgraph.store.base import BaseStore


In [11]:
llm = ChatOllama(model='qwen2.5:1.5b',temperature=0.2,max_token=100)
embeddings = OllamaEmbeddings(model="nomic-embed-text:v1.5")

In [12]:
store = InMemoryStore(index={'embed':embeddings,'dims':1536})

In [13]:
# Everything under this namespace belongs to this customer
CUSTOMER_ID = "cust_001"
MEMORY_NAMESPACE = (CUSTOMER_ID, "preferences")

In [14]:
# pre-seeded long term memories
store.put(
    MEMORY_NAMESPACE,
    "1",
    {
        "text": "I prefer casual and comfortable clothing for everyday wear."
    }
)

store.put(
    MEMORY_NAMESPACE,
    "2",
    {
        "text": "My favorite brands are Nike, Gap, and Uniqlo."
    }
)

store.put(
    MEMORY_NAMESPACE,
    "3",
    {
        "text": "My budget is typically between $50-100 per item."
    }
)

store.put(
    MEMORY_NAMESPACE,
    "4",
    {
        "text": "I usually wear size M tops and 32 inch jeans."
    }
)

In [15]:
class State(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]
    memories: list[str]

In [16]:
def retrieve_memories(state:State,store:BaseStore):
    user_query = state['messages'][-1].content
    res = store.search(
        MEMORY_NAMESPACE,
        query=user_query,
        limit=5
    )
    memories = []
    for result in res:
        memories.append(result.value['text'])
    return {'memories':memories}

def generate_response(state:State,store:BaseStore):
    user_query = state['messages'][-1].content
    memories = state.get('memories',[])
    memory_text = '\n'.join(f' - {memory}' for memory in memories)
    system_prompt = f"""
        You are an e-commerce shopping assistant.
    
        The following are long-term memories about the customer:
    
        {memory_text}
    
        Use these memories together with the current conversation
        to provide personalized fashion recommendations.
    
        Do not claim to remember conversation messages.
        Only use the long-term customer preferences provided above.
    
        Current customer query:
        {user_query}
        """
    res = llm.invoke([
        SystemMessage(content=system_prompt),
        *state['messages']
    ])
    return {'messages':[res]}

def save_memory(state:State,store:BaseStore):
    user_message = None
    for message in reversed(state['messages']):
        if isinstance(message,HumanMessage):
            user_message = message
            break

    if user_message:
        memory_id = str(uuid.uuid4())
        store.put(
            MEMORY_NAMESPACE,
            memory_id,
            {
                "text":f'Customer inquiry : {user_message}'  
            } 
        )
    return {}




In [17]:
graph = StateGraph(State)
graph.add_node("retrieve_memories",retrieve_memories)
graph.add_node("generate_response",generate_response)
graph.add_node("save_memory",save_memory)

graph.add_edge(START,"retrieve_memories")
graph.add_edge("retrieve_memories","generate_response")
graph.add_edge("generate_response","save_memory")
graph.add_edge("save_memory",END)

checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer,store=store)

In [18]:
print(workflow.get_graph().draw_ascii())

    +-----------+      
    | __start__ |      
    +-----------+      
          *            
          *            
          *            
+-------------------+  
| retrieve_memories |  
+-------------------+  
          *            
          *            
          *            
+-------------------+  
| generate_response |  
+-------------------+  
          *            
          *            
          *            
   +-------------+     
   | save_memory |     
   +-------------+     
          *            
          *            
          *            
     +---------+       
     | __end__ |       
     +---------+       


In [19]:
config_thread_1 = {"configurable": {"thread_id": "thread_1"}}
result_1 = workflow.invoke(
    {
        "messages": [HumanMessage(content="I am looking for running shoes.")]
    },
    config=config_thread_1
)
print("THREAD 1:",result_1["messages"][-1].content)

THREAD 1: Based on your preference for casual and comfortable clothing, you might enjoy these brands: Nike, Gap, and Uniqlo. They offer a variety of styles that can fit within your budget range of $50-100 per item. For running shoes specifically, consider the following options:

1. **Nike** - Known for their lightweight and breathable designs.
2. **Gap** - Offers both casual and athletic wear, including some comfortable running shoe models.
3. **Uniqlo** - Often offers affordable yet high-quality sneakers that are perfect for everyday use.

Would you like more information on any of these brands or would you prefer to browse through their specific collections?


In [20]:
config_thread_1 = {"configurable": {"thread_id": "thread_1"}}
result_2 = workflow.invoke(
    {
        "messages": [HumanMessage(content="I prefer Nike running shoes.")]
    },
    config=config_thread_1
)
print("THREAD 1 result 2:",result_2["messages"][-1].content)

THREAD 1 result 2: Great choice! Here's a quick look at some popular Nike running shoes:

- **Nike Air Max 270 Running Shoes**: These are lightweight and feature the iconic Air Max cushioning, making them perfect for everyday wear.
- **Nike Free Run 1.1**: Ideal if you prefer more comfort with minimal support, these shoes offer a great balance between style and functionality.
- **Nike React Infinity Runner 18**: Known for their responsive and comfortable ride, these are excellent choices for runners who prioritize comfort.

Would you like to see the latest styles or do you have any specific features in mind?


In [21]:
config_thread_2 = {"configurable": {"thread_id": "thread_2"}}
result_2 = workflow.invoke(
    {
        "messages": [HumanMessage(content="Suggest some running shoes for me.")]
    },
    config=config_thread_2
)
print("THREAD 2:",result_2["messages"][-1].content)

THREAD 2: Based on your preference for Nike, Gap, and Uniqlo, I recommend checking out their latest running shoe collections. Here are a few options:

1. **Nike** - They have a variety of running shoes that cater to different styles and preferences. Look into the Air Zoom Pegasus or the Flyknit 27.
   
2. **Gap** - Gap offers both casual and athletic footwear, including their "Athleisure" line which is perfect for everyday wear but still suitable for running. Consider the "Racerback" shoes.

3. **Uniqlo** - Uniqlo's running shoes are designed to be comfortable and functional. They have a range of models that offer both style and performance, such as the "WALKER" series.

These brands should provide you with a good selection of running shoes that fit your preferences.


In [22]:
config_thread_1 = {"configurable": {"thread_id": "thread_1"}}
for i in workflow.stream({"messages": [HumanMessage(content="I am looking for running shoes.")]},config=config_thread_1):
    print(i)
    

{'retrieve_memories': {'memories': ["Customer inquiry : content='I am looking for running shoes.' additional_kwargs={} response_metadata={} id='8b632c35-0f4f-4c27-b2f2-580a789e268e'", "Customer inquiry : content='Suggest some running shoes for me.' additional_kwargs={} response_metadata={} id='a9ef6172-162a-47fc-98e1-ddc2d7f47ecd'", "Customer inquiry : content='I prefer Nike running shoes.' additional_kwargs={} response_metadata={} id='9e711dbe-0759-4598-9839-6b111f893cfd'", 'My favorite brands are Nike, Gap, and Uniqlo.', 'I prefer casual and comfortable clothing for everyday wear.']}}
{'generate_response': {'messages': [AIMessage(content="Got it! Here's a quick look at some popular Nike running shoes:\n\n- **Nike Air Max 270 Running Shoes**: Lightweight and with iconic Air Max cushioning, these are perfect for everyday wear.\n- **Nike Free Run 1.1**: Offers comfort without sacrificing support, making them great for those who prefer more flexibility in their routine.\n- **Nike React I